# Intelligent Support Router & Triage

This notebook builds an end-to-end NLP system for classifying customer support tickets using TensorFlow, Hugging Face Transformers, and BERT.

Categories:
- Urgent Complaint
- Feature Suggestion
- General Inquiry

## 2. Environment Setup
Install only the required packages and keep versions compatible with TensorFlow and Hugging Face.

In [ ]:
!pip uninstall -y tensorflow tensorflow-text tf_keras keras keras-hub keras-nlp
!pip install pyngrok --quiet

!pip install \
tensorflow==2.20.0 \
tensorflow-text==2.20.0 \
tf-keras==2.20.0 \
keras==3.10.0 \
protobuf==5.29.5 \
numpy==2.0.2 \
--quiet

!pip uninstall -y transformers
!pip install transformers==4.52.4 --quiet


In [7]:
## 3. Import Libraries
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import transformers
from flask import Flask, request, render_template_string
from pyngrok import ngrok
from sklearn.metrics import classification_report, confusion_matrix
from transformers import pipeline, AutoTokenizer, TFAutoModelForSequenceClassification

# Initialize the web application framework
app = Flask(__name__)

# Convert relative path to an absolute path to bypass Hugging Face repo-ID validation
MODEL_PATH = os.path.abspath("./saved_tf_bert_model")
# 4. Dataset Creation
## Create a balanced custom dataset with three classes:
## - Urgent Complaint
## - Feature Suggestion
## - General Inquiry

## The dataset was expanded from 18 samples to 90 samples to improve generalization.
data = {
    "text": [
        # Class 0: Urgent Complaints
        "My account is locked and I cannot access my business dashboard.",
        "The checkout system crashed and customers cannot complete payments.",
        "Your software is down and our team cannot process orders.",
        "The database went offline and we are losing customer records.",
        "Someone accessed my admin account without permission.",
        "Payments are failing for all users on the live application.",
        "I was charged twice and need this fixed immediately.",
        "The app keeps crashing every time I upload a file.",
        "My subscription was canceled even though I paid this month.",
        "The login page is broken and none of our employees can sign in.",
        "We are seeing duplicate invoices for multiple customers.",
        "Our production system is unavailable after the latest update.",
        "I cannot reset my password and I am locked out.",
        "The report export failed and deleted important data.",
        "Your platform is returning server errors for every request.",
        "A security alert appeared and I need urgent help.",
        "Our customers are unable to place orders right now.",
        "The billing system charged the wrong credit card.",
        "The website freezes whenever users submit the form.",
        "My account shows incorrect payment information.",
        "The system is extremely slow and blocking our daily operations.",
        "I lost access to all project files after the update.",
        "The API is returning errors and our integration is broken.",
        "My team cannot access the workspace this morning.",
        "The application is not loading for any user in our company.",
        "I need immediate help because our client portal is offline.",
        "The invoice amount is incorrect and I need a refund.",
        "The upload feature is broken and stopping our workflow.",
        "We cannot access customer tickets after the maintenance window.",
        "The platform is failing during checkout and we are losing sales.",

        # Class 1: Feature Suggestions
        "It would be great to add dark mode to the dashboard.",
        "Please add an option to export reports to Excel.",
        "Can you add drag and drop file upload in the next update?",
        "I would like a dark mode toggle button.",
        "Please add the ability to export data to CSV and PDF.",
        "Multi-language support would be very helpful.",
        "Can you add email notifications for new tickets?",
        "It would be useful to customize dashboard widgets.",
        "Please add a mobile app for Android and iOS.",
        "I suggest adding Slack integration for alerts.",
        "A search bar inside the documentation would help a lot.",
        "Can you add keyboard shortcuts for power users?",
        "It would be nice to schedule automatic reports.",
        "Please add role-based permissions for team members.",
        "I would like to save custom filters on the tickets page.",
        "Can you add a calendar view for scheduled tasks?",
        "Please include AI suggestions for support replies.",
        "It would be helpful to integrate with Google Drive.",
        "Can you add a bulk upload option for customer data?",
        "I suggest adding a notification center inside the app.",
        "Please add more chart types to the analytics page.",
        "Can users personalize the sidebar layout?",
        "It would help if we could pin important tickets.",
        "Please add templates for common customer responses.",
        "I would like a feature to archive old projects automatically.",
        "Can you add export options for audit logs?",
        "Please add support for webhooks in the settings page.",
        "It would be helpful to tag tickets automatically.",
        "Can you add a voice input option for notes?",
        "Please add a preview mode before publishing changes.",

        # Class 2: General Inquiries
        "Hello, what are your pricing plans?",
        "Where can I find the API documentation?",
        "Do you support single sign-on?",
        "Can you send me information about subscription costs?",
        "Where is the setup guide for developers?",
        "How does your team handle basic customer support?",
        "Do you offer a free trial?",
        "What payment methods do you accept?",
        "Can I upgrade my plan later?",
        "Where can I find the billing history page?",
        "Do you have documentation for account setup?",
        "What are your business hours?",
        "Can you explain the difference between your plans?",
        "Do you provide student discounts?",
        "Where can I download invoices?",
        "How many users are included in the standard plan?",
        "Do you support enterprise accounts?",
        "Can I schedule a demo with your sales team?",
        "Where can I learn about your security policy?",
        "How do I contact customer support?",
        "Is there a limit on API requests?",
        "Can I change my email address?",
        "Where do I update my company profile?",
        "Do you have onboarding materials?",
        "How long does account verification take?",
        "Can I invite more team members?",
        "Where can I read the refund policy?",
        "Do you provide training sessions?",
        "How do I connect my account to third-party tools?",
        "Can you explain how your support ticket system works?"
    ],
    "label": (
        [0] * 30 +
        [1] * 30 +
        [2] * 30
    )

}
## Model Training, Evaluation & Persistence
## This section fine-tunes the pretrained BERT model using TensorFlow, evaluates its performance using a Classification Report and Confusion Matrix, and saves the trained model and tokenizer for deployment in the Flask application.
df = pd.DataFrame(data)
model_checkpoint = "google/bert_uncased_L-2_H-128_A-2"

# 2. Tokenize the structured training text
train_tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokens = train_tokenizer(list(df["text"]), padding=True, truncation=True, max_length=64, return_tensors="tf")

# 3. Instantiate the TensorFlow Base Architecture for Transfer Learning Finetuning
train_model = TFAutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=3)

# 4. Define optimization algorithms and compilation variables
optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
train_model.compile(optimizer=optimizer, loss=loss, metrics=["accuracy"])

# 5. Fine-tune the pre-trained transformer layers
labels = tf.convert_to_tensor(df["label"])
train_model.fit(dict(tokens), labels, epochs=3, batch_size=2, verbose=0)

# 6. Generate validation predictions for downstream evaluation metrics matrices
raw_predictions = train_model.predict(dict(tokens), verbose=0)
y_pred = np.argmax(raw_predictions.logits, axis=-1)
y_true = df["label"].values

print("\n" + "="*50)
print("📊 REQUIREMENT D: CLASSIFICATION REPORT")
print("="*50)
print(classification_report(y_true, y_pred, target_names=["Urgent Complaint", "Feature Suggestion", "General Inquiry"]))

print("="*50)
print("📊 REQUIREMENT D: CONFUSION MATRIX")
print("="*50)
print(confusion_matrix(y_true, y_pred))
print("="*50 + "\n")

# 7. Persist fine-tuned model artifacts smoothly to absolute disk path
print(f"Saving fine-tuned model artifacts locally to: {MODEL_PATH}")
os.makedirs(MODEL_PATH, exist_ok=True)
train_model.save_pretrained(MODEL_PATH)
train_tokenizer.save_pretrained(MODEL_PATH)
print("Model training lifecycle complete!")

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



📊 REQUIREMENT D: CLASSIFICATION REPORT
                    precision    recall  f1-score   support

  Urgent Complaint       0.97      1.00      0.98        30
Feature Suggestion       1.00      0.87      0.93        30
   General Inquiry       0.91      1.00      0.95        30

          accuracy                           0.96        90
         macro avg       0.96      0.96      0.95        90
      weighted avg       0.96      0.96      0.95        90

📊 REQUIREMENT D: CONFUSION MATRIX
[[30  0  0]
 [ 1 26  3]
 [ 0  0 30]]

Saving fine-tuned model artifacts locally to: /content/saved_tf_bert_model
Model training lifecycle complete!


In [9]:
## ## Model Initialization

## Load the fine-tuned BERT model and tokenizer for inference. If a locally fine-tuned model is unavailable, the system automatically falls back to the pretrained BERT checkpoint. Finally, initialize the Qwen generative language model to draft intelligent responses after ticket classification.
print("Initializing model architecture layers...")

# A. Evaluate and load the newly fine-tuned classification model component
if os.path.exists(MODEL_PATH) and any(os.scandir(MODEL_PATH)):
    print(f"-> Loading local fine-tuned BERT model from: {MODEL_PATH}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = TFAutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
else:
    print("-> Local fine-tuned model folder not found.")
    print("-> Falling back to the base BERT checkpoint (Module 4, Session 18 setup).")
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
    model = TFAutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=3)

# B. Initialize the Generative LLM component for response drafting
print("-> Loading Qwen text-generation pipeline...")
generation_pipeline = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device_map="auto"
)
print("-> All model frameworks ready.")

Initializing model architecture layers...
-> Loading local fine-tuned BERT model from: /content/saved_tf_bert_model


Some layers from the model checkpoint at /content/saved_tf_bert_model were not used when initializing TFBertForSequenceClassification: ['dropout_15']
- This IS expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertForSequenceClassification were initialized from the model checkpoint at /content/saved_tf_bert_model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertForSequenceClassification for predictions without further training.


-> Loading Qwen text-generation pipeline...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


-> All model frameworks ready.


## User Interface Layout Dashboard Design

In [10]:
HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>AI Support Router Dashboard</title>
    <style>
        body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 40px; background-color: #f4f6f9; color: #333; }
        .container { max-width: 700px; background: white; padding: 30px; border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.08); margin: 0 auto; }
        h2 { color: #007bff; margin-top: 0; }
        p.subtitle { color: #666; font-size: 14px; margin-bottom: 25px; }
        textarea { width: 100%; height: 120px; padding: 12px; margin-bottom: 15px; border-radius: 5px; border: 1px solid #ccc; box-sizing: border-box; font-size: 14px; resize: vertical; }
        button { background-color: #007bff; color: white; border: none; padding: 12px 24px; border-radius: 5px; cursor: pointer; font-size: 14px; font-weight: bold; width: 100%; transition: background-color 0.2s; }
        button:hover { background-color: #0056b3; }
        .result { margin-top: 25px; padding: 20px; background: #e9ecef; border-radius: 5px; border-left: 5px solid #007bff; }
        .result h3 { margin-top: 0; font-size: 18px; }
        .intent-badge { background-color: #28a745; color: white; padding: 4px 10px; border-radius: 20px; font-size: 14px; display: inline-block; margin-left: 5px; }
        .response-box { background: white; padding: 15px; border-radius: 4px; border: 1px solid #ddd; font-family: monospace; white-space: pre-wrap; margin-top: 10px; font-size: 13px; }
    </style>
</head>
<body>
    <div class="container">
        <h2>🏢 Intelligent Support Router & Triage</h2>
        <p class="subtitle">Advanced AI Diploma Capstone Project — Classification & Generation Pipeline</p>

        <form method="POST" action="/process">
            <textarea name="email_text" placeholder="Paste incoming customer email or ticket details here..." required></textarea>
            <button type="submit">Route & Process Ticket</button>
        </form>

        {% if intent %}
        <div class="result">
            <h3>Routed Category: <span class="intent-badge">{{ intent }}</span></h3>
            <p><strong>Drafted System Action / AI Response:</strong></p>
            <div class="response-box">{{ response }}</div>
        </div>
        {% endif %}
    </div>
</body>
</html>
"""

## Web Endpoints & Machine Learning Core Routing Logic

In [11]:
@app.route('/', methods=['GET'])
def home():
    """Renders the main dashboard user interface."""
    return render_template_string(HTML_TEMPLATE)

@app.route('/process', methods=['POST'])
def process():
    """Processes incoming text inputs, runs inference, and requests an LLM generation response."""
    user_text = request.form.get('email_text', '').strip()

    if not user_text:
        return render_template_string(HTML_TEMPLATE, intent="Error", response="Please enter valid text.")

    # 1. Tokenization and Numeric Transformation
    inputs = tokenizer(user_text, return_tensors="tf", padding=True, truncation=True, max_length=64)

    # 2. TensorFlow Classification Inference
    outputs = model(inputs)
    probabilities = tf.nn.softmax(outputs.logits, axis=-1)
    predicted_class = tf.argmax(probabilities, axis=-1).numpy()[0]

    # Map indices to explicit curriculum categories
    class_labels = {0: "Urgent Complaint", 1: "Feature Suggestion", 2: "General Inquiry"}
    detected_intent = class_labels.get(predicted_class, "General Inquiry")

    # 3. Conditional Prompt Configuration Selection
    system_prompts = {
        0: "You are an urgent customer response assistant. Draft an explicit, brief internal escalation note for a manager outlining the critical issue.",
        1: "You are a product management assistant. Extract the product feedback from this email and format it into a concise feature request list.",
        2: "You are a polite technical desk support agent. Draft a professional, one-sentence confirmation response confirming receipt of their inquiry."
    }

    # Format structured chat blocks for the Chat Template engine
    messages = [
        {"role": "system", "content": system_prompts.get(predicted_class, system_prompts[2])},
        {"role": "user", "content": user_text}
    ]

    # 4. LLM Generation Construction
    prompt = generation_pipeline.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    gen_outputs = generation_pipeline(prompt, max_new_tokens=120, do_sample=True, temperature=0.4)

    # Isolate assistant text by filtering template markers cleanly
    ai_response = gen_outputs[0]["generated_text"].split("<|im_start|>assistant\n")[-1].strip()

    return render_template_string(HTML_TEMPLATE, intent=detected_intent, response=ai_response)

## Step 6: Ngrok Tunnel Generation & Server Startup

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

NGROK_TOKEN = userdata.get("NGROK_AUTH_TOKEN")
ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(5000)

print("="*60)
print("🌍 PUBLIC EVALUATION URL:", public_url.public_url)
print("="*60)

app.run(host="0.0.0.0", port=5000)

🌍 PUBLIC EVALUATION URL: https://nerd-overstock-exciting.ngrok-free.dev
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [28/Jun/2026 09:11:34] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [28/Jun/2026 09:11:35] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [28/Jun/2026 09:11:56] "POST /process HTTP/1.1" 200 -
